# Coarse-Grained Animal Classification

This notebook performs coarse-grained classification across 10 animal classes using:
- provided image features,
- handcrafted image features,
- pretrained ResNet18 embeddings,
- and several supervised classification models.

The workflow compares feature representations, compares classification models using stratified 5-fold cross-validation, tunes the best model, and evaluates it on a holdout validation set.

In [ ]:
# library 
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path("../data/task1")
OUTPUT_DIR = Path("../results/task1")
CACHE_DIR = Path("../cache/task1")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "task1_predictions.csv"

RANDOM_STATE = 42

In [ ]:
DATA_DIR = Path("../data/task1")
OUTPUT_DIR = Path("../results/task1")
CACHE_DIR = Path("../cache/task1")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "task1_predictions.csv"

RANDOM_STATE = 42

In [ ]:
# load metadata and provided feature csv files, then merge them into 1 dataframe
def load_csv_features(data_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # read train and test metadata
    train_meta = pd.read_csv(data_dir / "train_metadata.csv")
    test_meta = pd.read_csv(data_dir / "test_metadata.csv")

    # combine train and test metadata
    # test set has no label, so fill with NaN 
    all_meta = pd.concat(
        [
            train_meta[["image_id", "image_path", "class_id", "class_name"]],
            test_meta.assign(class_id = np.nan, class_name = np.nan)[
                ["image_id", "image_path", "class_id", "class_name"]
            ],
        ],
        ignore_index=True
    )

    features = all_meta.copy()

    # merge all provided features using image_id 
    for filename in [
        "color_histogram.csv",
        "hog_pca.csv",
        "additional_features.csv"
    ]: 
        feature_df = pd.read_csv(data_dir / filename)
        features = features.merge(feature_df, on = "image_id", how="left")

    return features, train_meta



In [ ]:
def engineer_image_features(data_dir: Path, metadata: pd.DataFrame) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []
    
    """
    create additional features directly from images, including: 
    - RGB statistics
    - grayscale statistics 
    - color contrast
    - centre vs border brightness
    - local grid brightness
    """

    # process each image
    for _, row in metadata.iterrows():
        image_path = data_dir / row["image_path"]

        # convert image to RGB and resize to 64x64 
        img = Image.open(image_path).convert("RGB").resize((64, 64))

        # convert image to numpy array and normalise pixel values 
        arr = np.asarray(img).astype(np.float32) / 255.0

        # split RGB channels 
        r,g,b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]

        # create grayscale image 
        gray = 0.299 * r + 0.587 * g + 0.114 * b 

        feature_dict = {"image_id": row["image_id"]}

        # RGB statistical features
        for name, channel in zip(["r", "g", "b"], [r, g, b]):
            feature_dict[f"{name}_mean"] = float(channel.mean())
            feature_dict[f"{name}_std"] = float(channel.std())

            # quantiles capture colour distribution
            feature_dict[f"{name}_q25"] = float(np.quantile(channel, 0.25))
            feature_dict[f"{name}_q50"] = float(np.quantile(channel, 0.50))
            feature_dict[f"{name}_q75"] = float(np.quantile(channel, 0.75))

        # Color contrast features 
        # red-green difference 
        rg = r - g

        # yellow-blue difference 
        yb = 0.5 * (r + g) - b 

        feature_dict["rg_mean_abs"] = float(np.mean(np.abs(rg)))
        feature_dict["yb_mean_abs"] = float(np.mean(np.abs(yb)))

        feature_dict["rg_std"] = float(np.std(rg))
        feature_dict["yb_std"] = float(np.std(yb))

        # Grayscale features 
        feature_dict["gray_mean"] = float(gray.mean())
        feature_dict["gray_std"] = float(gray.std())

        # Centre vs border brightness
        # extract centre region 
        centre = gray[16:48, 16:48]

        # create border mask 
        border_mask = np.ones_like(gray, dtype=bool)
        border_mask[16:48, 16:48] = False
        border = gray[border_mask]

        feature_dict["centre_mean"] = float(centre.mean())
        feature_dict["border_mean"] = float(border.mean())

        # difference between object centre and background 
        feature_dict["centre_border_diff"] = float(centre.mean() - border.mean())

        # Local brightness and grid features 
        # divide image into 4x4 regions 
        cell = 16
        for i in range(4):
            for j in range(4):
                patch = gray[
                    i * cell:(i + 1) * cell,
                    j * cell:(j + 1) * cell
                ]

                feature_dict[f"grid_gray_{i}_{j}"] = float(patch.mean())

        rows.append(feature_dict)
    return pd.DataFrame(rows)

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# remove final classifier, so model outputs 512-dimensional features
resnet.fc = nn.Identity()

resnet = resnet.to(device)
resnet.eval()

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class ImageFeatureDataset(Dataset):
    def __init__(self, metadata, data_dir):
        self.metadata = metadata.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        image_path = self.data_dir / row["image_path"]
        image = Image.open(image_path).convert("RGB")
        image = resnet_transform(image)

        return image, row["image_id"]

In [ ]:
def extract_resnet_features(metadata, data_dir, batch_size=64):
    dataset = ImageFeatureDataset(metadata, data_dir)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    all_features = []
    all_ids = []

    with torch.no_grad():
        for images, image_ids in loader:
            images = images.to(device)

            features = resnet(images)
            features = features.cpu().numpy()

            all_features.append(features)
            all_ids.extend(image_ids)

    all_features = np.vstack(all_features)

    feature_df = pd.DataFrame(
        all_features,
        columns=[f"resnet_{i}" for i in range(all_features.shape[1])]
    )

    feature_df.insert(0, "image_id", all_ids)

    return feature_df

In [ ]:
train_meta = pd.read_csv(DATA_DIR / "train_metadata.csv")
test_meta = pd.read_csv(DATA_DIR / "test_metadata.csv")

all_meta = pd.concat(
    [
        train_meta[["image_id", "image_path"]],
        test_meta[["image_id", "image_path"]]
    ],
    ignore_index=True
)

resnet_cache_path = DATA_DIR / "resnet18_features_task1.csv"

if resnet_cache_path.exists():
    resnet_features = pd.read_csv(resnet_cache_path)
else:
    resnet_features = extract_resnet_features(
        all_meta,
        DATA_DIR,
        batch_size=64
    )

    resnet_features.to_csv(
        resnet_cache_path,
        index=False
    )

resnet_features.head()

In [ ]:
def prepare_data(data_dir: Path, resnet_features: pd.DataFrame, feature_mode="all"):
    """
    Prepare data using different feature sets.

    feature_mode options:
    - "provided_engineered": provided CSV features + handcrafted engineered image features
    - "resnet_only": ResNet18 features only
    - "all": provided CSV features + engineered features + ResNet18 features
    """

    features, train_meta = load_csv_features(DATA_DIR)

    # cache engineered features
    cache_path = data_dir / "engineered_image_features_task1.csv"

    if cache_path.exists():
        image_features = pd.read_csv(cache_path)
    else:
        image_features = engineer_image_features(
            data_dir,
            features[["image_id", "image_path"]]
        )
        image_features.to_csv(cache_path, index=False)

    # keep metadata columns
    metadata_cols = ["image_id", "image_path", "class_id", "class_name"]

    if feature_mode == "provided_engineered":

        # provided CSV features + handcrafted engineered features
        features = features.merge(
            image_features,
            on="image_id",
            how="left"
        )

    elif feature_mode == "resnet_only":

        # remove provided feature columns and use only ResNet features
        features = features[metadata_cols].merge(
            resnet_features,
            on="image_id",
            how="left"
        )

    elif feature_mode == "all":

        # provided CSV features + engineered features + ResNet features
        features = features.merge(
            image_features,
            on="image_id",
            how="left"
        )

        features = features.merge(
            resnet_features,
            on="image_id",
            how="left"
        )

    else:
        raise ValueError(
            "feature_mode must be one of: "
            "'provided_engineered', 'resnet_only', or 'all'"
        )

    train_df = features[features["class_id"].notna()].copy()
    test_df = features[features["class_id"].isna()].copy()

    feature_cols = [
        c for c in features.columns
        if c not in ["image_id", "image_path", "class_id", "class_name"]
    ]

    X = train_df[feature_cols].fillna(0.0).values
    y = train_df["class_id"].astype(int).values
    X_test = test_df[feature_cols].fillna(0.0).values

    class_lookup = (
        train_meta[["class_id", "class_name"]]
        .drop_duplicates()
        .sort_values("class_id")
        .set_index("class_id")["class_name"]
        .to_dict()
    )

    test_ids = test_df[["image_id"]].reset_index(drop=True)

    return X, y, X_test, test_ids, class_lookup

In [ ]:
def build_models() -> Dict[str, object]:
    models = {}

    models["logistic_regression"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    models["random_forest"] = RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=1,
        random_state=RANDOM_STATE
    )

    models["lightgbm"] = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        verbose=-1,
        force_col_wise=True
)

    models["xgboost"] = XGBClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=1
    )

    models["catboost"] = CatBoostClassifier(
        iterations=250,
        learning_rate=0.05,
        depth=6,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        random_state=RANDOM_STATE,
        verbose=0
    )

    models["rbf_svm"] = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("clf", SVC(
            kernel="rbf",
            C=10.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE
        ))
    ])

    models["ensemble_xgb_cb_rbf"] = VotingClassifier(
        estimators=[
            ("xgb", clone(models["xgboost"])),
            ("cat", clone(models["catboost"])),
            ("rbf", clone(models["rbf_svm"]))
        ],
        voting="soft",
        weights=[3, 3, 2],
        n_jobs=1
    )

    return models

In [ ]:
# compare models using stratified 5-fold cross validation 
def evaluate_models(X, y, models):
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    results = []
    for name, model in models.items():
        # Compute cross-validation accuracy
        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="accuracy",
            n_jobs=1
        )

        results.append({
            "model": name,
            "mean_cv_accuracy": scores.mean(),
            "std_cv_accuracy": scores.std(),
            "fold_scores": np.round(scores, 4).tolist(),
        })

    return pd.DataFrame(results).sort_values(
        "mean_cv_accuracy",
        ascending=False
    )

In [ ]:
# Train on training split and evaluate on validation split 

def validation_report(
    X,
    y,
    model,
    class_lookup,
    show_confusion=True,
    show_top_confused=True,
    top_n=5
):
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=RANDOM_STATE
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    labels = sorted(class_lookup.keys())
    target_names = [class_lookup[i] for i in labels]

    acc = accuracy_score(y_val, pred)

    print("Holdout validation accuracy:")
    print(round(acc, 4))

    print("\nClassification report:")
    print(classification_report(
        y_val,
        pred,
        labels=labels,
        target_names=target_names
    ))

    cm = confusion_matrix(y_val, pred, labels=labels)

    cm_df = pd.DataFrame(
        cm,
        index=target_names,
        columns=target_names
    )

    if show_confusion:
        plt.figure(figsize=(9, 7))
        sns.heatmap(
            cm_df,
            annot=True,
            fmt="d",
            cmap="Blues"
        )
        plt.title("Confusion Matrix")
        plt.ylabel("True class")
        plt.xlabel("Predicted class")
        plt.tight_layout()
        plt.show()

    if show_top_confused:
        cm_no_diag = cm.copy()
        np.fill_diagonal(cm_no_diag, 0)

        confused_pairs = []

        for true_idx in range(cm_no_diag.shape[0]):
            for pred_idx in range(cm_no_diag.shape[1]):
                count = cm_no_diag[true_idx, pred_idx]

                if count > 0:
                    confused_pairs.append({
                        "true_class": target_names[true_idx],
                        "predicted_class": target_names[pred_idx],
                        "count": count
                    })

        confused_df = pd.DataFrame(confused_pairs)

        if not confused_df.empty:
            confused_df = confused_df.sort_values(
                "count",
                ascending=False
            ).head(top_n)

            print(f"\nTop {top_n} confused pairs:")
            print(confused_df)
        else:
            print("\nNo misclassified pairs found.")

    return acc, cm_df

In [ ]:
# Train final model on full training data and generate Kaggle predictions
def save_kaggle_predictions(model, X, y, X_test, test_ids, output_path):

    # train model on full training set
    model.fit(X, y)

    # Predict test labels
    test_pred_ids = model.predict(X_test).astype(int)

    # Create submission dataframe
    submission = test_ids.copy()
    submission["class_id"] = test_pred_ids

    # Save CSV
    submission.to_csv(output_path, index=False)

    print(f"\nSaved predictions to: {output_path}")
    print(submission.head())